# N01: Extensive Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import nibabel as nib
from concurrent.futures import ThreadPoolExecutor

# Set plot style
plt.style.use('ggplot')
os.makedirs('/kaggle/working/eda_outputs', exist_ok=True)


## Configuration & Data Loading
Check if we are loading from a previously saved Kaggle notebook (N00) dataset, or locally.

In [ ]:
# Check for N00 output in Kaggle input datasets
# Replace 'your-n00-dataset-name' with the actual folder name Kaggle gives to your N00 output!
N00_DATA_DIR = '/kaggle/input/notebooks/assemelqirsh/n00-path-exploration'

# Fallback to working directory if running in same notebook or locally
if not os.path.exists(os.path.join(N00_DATA_DIR, 'mri_file_paths.csv')):
    N00_DATA_DIR = '/kaggle/working'
    if not os.path.exists('mri_file_paths.csv') and not os.path.exists('/kaggle/working/mri_file_paths.csv'):
        # Maybe it's just in current dir
        N00_DATA_DIR = '.'

print(f"Using DATA_DIR: {N00_DATA_DIR}")

df_path = os.path.join(N00_DATA_DIR, 'mri_file_paths.csv')
paired_path = os.path.join(N00_DATA_DIR, 'paired_mri_data_master.csv')

print("Loading data...")
df = pd.read_csv(df_path)
wide_df = pd.read_csv(paired_path) if os.path.exists(paired_path) else None

# Extract metadata
def parse_filename(path):
    basename = os.path.basename(path).replace('.nii.gz', '').replace('.nii', '')
    parts = basename.split('_')
    if len(parts) >= 4:
        return parts[0], parts[1], parts[2], parts[3]
    return None, None, None, None

df['cohort'], df['modality'], df['field_strength'], df['subject_id'] = zip(*df['file_path'].map(parse_filename))
df = df.dropna(subset=['subject_id'])


## Dataset Completeness & Basic Statistics

In [ ]:
print("
--- 1. Dataset Completeness ---")
print(f"Total valid files: {len(df)}")
print(f"Total unique subjects: {df['subject_id'].nunique()}")

# Plot 1: Counts by Field Strength & Modality
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='field_strength', hue='modality', order=['0.1T', '1.5T', '3T', '5T', '7T'])
plt.title("Distribution of Scans by Field Strength and Modality")
plt.ylabel("Number of Scans")
plt.xlabel("Field Strength")
plt.legend(title='Modality')
plt.tight_layout()
plt.savefig('/kaggle/working/eda_outputs/01_distribution.png', dpi=300)
plt.show()

if wide_df is not None:
    print("
--- Pairing Analysis (Paired vs Unpaired) ---")
    field_cols = [c for c in ['0.1T', '1.5T', '3T', '5T', '7T'] if c in wide_df.columns]
    
    overlap_counts = wide_df[field_cols].notna().sum(axis=1)
    plt.figure(figsize=(8, 5))
    sns.countplot(x=overlap_counts)
    plt.title("Number of Available Field Strengths per Subject")
    plt.xlabel("Number of Available Scans (out of 5 possible field strengths)")
    plt.ylabel("Number of Subjects")
    plt.tight_layout()
    plt.savefig('/kaggle/working/eda_outputs/02_pairing_overlap.png', dpi=300)
    plt.show()
    
    print("Subject Pairing Distribution:")
    print(overlap_counts.value_counts().sort_index())
    print("NOTE: If most subjects only have 1 scan, this challenge focuses heavily on UNPAIRED translation (e.g. CycleGAN).")


## Physical Volume Characteristics & Intensities

In [ ]:
print("
--- 2. Physical Characteristics ---")
RUNNING_ON_KAGGLE = os.path.exists('/kaggle/input') 

if not RUNNING_ON_KAGGLE:
    print("[!] Not running on Kaggle. Skipping physical file inspection (nibabel) since the .nii files are unavailable locally.")
else:
    print("Extracting physical properties from a sample of NIfTI files... (This may take a minute)")
    
    sample_df = df.groupby('field_strength').apply(lambda x: x.sample(min(len(x), 50))).reset_index(drop=True)
    
    shapes, spacings, max_intensities = [], [], []
    
    def process_file(row):
        path = row['file_path']
        try:
            img = nib.load(path)
            header = img.header
            shape = img.shape
            spacing = header.get_zooms()
            
            data = img.get_fdata()
            p99 = np.percentile(data, 99)
            
            return {
                'field_strength': row['field_strength'],
                'modality': row['modality'],
                'shape_x': shape[0], 'shape_y': shape[1], 'shape_z': shape[2] if len(shape) > 2 else 1,
                'spacing_x': spacing[0], 'spacing_y': spacing[1], 'spacing_z': spacing[2] if len(spacing) > 2 else 1,
                'p99_intensity': p99
            }
        except Exception as e:
            return None
            
    results = []
    with ThreadPoolExecutor(max_workers=4) as executor:
        for res in executor.map(process_file, [row for _, row in sample_df.iterrows()]):
            if res:
                results.append(res)
                
    props_df = pd.DataFrame(results)
    
    if not props_df.empty:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=props_df, x='field_strength', y='spacing_z', hue='modality', order=['0.1T', '1.5T', '3T', '5T', '7T'])
        plt.title("Z-Axis Voxel Spacing (Slice Thickness) by Field Strength")
        plt.ylabel("Spacing (mm)")
        plt.tight_layout()
        plt.savefig('/kaggle/working/eda_outputs/03_z_spacing.png', dpi=300)
        plt.show()
        
        plt.figure(figsize=(10, 6))
        sns.violinplot(data=props_df, x='field_strength', y='p99_intensity', hue='modality', order=['0.1T', '1.5T', '3T', '5T', '7T'])
        plt.title("99th Percentile Intensity Distribution across Field Strengths")
        plt.ylabel("99th Percentile Voxel Intensity")
        plt.yscale('log')
        plt.tight_layout()
        plt.savefig('/kaggle/working/eda_outputs/04_intensity_dist.png', dpi=300)
        plt.show()
        
        print("Physical characteristics extraction complete.")
